In [2]:
import requests
import hashlib
import json
from datetime import datetime

# ────────────────────────────────────────────────
# Part 1: Simple Bitcoin Address Validator
# ────────────────────────────────────────────────
def is_valid_bitcoin_address(addr: str) -> bool:
    """Improved Bitcoin address validator (Bech32 + legacy support)"""
    addr = addr.strip().lower()
    
    if addr.startswith('bc1'):  # Bech32 / Bech32m (SegWit / Taproot)
        charset = 'qpzry9x8gf2tvdw0s3jn54khce6mua7l'
        # Check charset (after the 'bc1')
        if not all(c in charset for c in addr[4:]):
            return False
        # Reasonable length range for Bitcoin mainnet Bech32
        if not (38 <= len(addr) <= 90):
            return False
        # Optional: could add more (version byte, program length), but this catches most invalids
        return True
    
    elif addr.startswith(('1', '3')):  # Legacy P2PKH / P2SH (Base58Check)
        try:
            import base58  # only needed for legacy; pip install base58 if using
            decoded = base58.b58decode(addr)
            if len(decoded) != 25:
                return False
            payload = decoded[:21]
            checksum = decoded[21:]
            h1 = hashlib.sha256(payload).digest()
            h2 = hashlib.sha256(h1).digest()
            return checksum == h2[:4]
        except (ImportError, Exception):
            # Fallback rough check if base58 not available
            allowed = '123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz'
            return 26 <= len(addr) <= 35 and all(c in allowed for c in addr)
    
    return False

# ────────────────────────────────────────────────
# Part 2: Fetch address stats from mempool.space
# ────────────────────────────────────────────────
def get_address_stats(address: str):
    base_url = "https://mempool.space/api"
    endpoints = [
        f"/address/{address}",           # main stats
        f"/address/{address}/txs/chain", # confirmed txs (optional, paginated)
    ]
    
    data = {}
    try:
        # Main address stats
        resp = requests.get(f"{base_url}/address/{address}", timeout=10)
        resp.raise_for_status()
        data = resp.json()
        
        # Optional: get a few recent confirmed txs (limit to first page)
        txs_resp = requests.get(f"{base_url}/address/{address}/txs/chain", timeout=10)
        if txs_resp.status_code == 200:
            data['recent_txs'] = txs_resp.json()[:3]  # show only first 3 for demo
        
    except requests.exceptions.RequestException as e:
        return {"error": f"API request failed: {str(e)}"}
    
    return data

# ────────────────────────────────────────────────
# Part 3: Pretty-print the results
# ────────────────────────────────────────────────
def print_address_summary(address: str, data: dict):
    if "error" in data:
        print(f"\nError: {data['error']}")
        return
    
    print(f"\n{'═' * 60}")
    print(f"Bitcoin Address Report (mempool.space API) - {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}")
    print(f"Address: {address}")
    print(f"{'═' * 60}")
    
    chain = data.get("chain_stats", {})
    mempool = data.get("mempool_stats", {})
    
    funded = chain.get("funded_txo_sum", 0)
    spent  = chain.get("spent_txo_sum", 0)
    balance_sat = funded - spent
    balance_btc = balance_sat / 100_000_000
    
    print(f"Confirmed Balance : {balance_btc:,.8f} BTC  ({balance_sat:,} sat)")
    print(f"Total Received    : {chain.get('funded_txo_sum', 0) / 100_000_000:,.8f} BTC")
    print(f"Total Spent       : {chain.get('spent_txo_sum', 0) / 100_000_000:,.8f} BTC")
    print(f"Confirmed Tx Count: {chain.get('tx_count', 'N/A')}")
    print(f"Mempool Activity  : {mempool.get('tx_count', 0)} pending txs")
    
    if 'recent_txs' in data and data['recent_txs']:
        print("\nRecent Confirmed Transactions (first few):")
        for tx in data['recent_txs'][:3]:
            txid = tx.get('txid', '—')
            status = tx.get('status', {})
            confirmed = status.get('confirmed', False)
            height = status.get('block_height', '—')
            print(f" • {txid[:12]}...  confirmed={confirmed}  height={height}")

    print(f"{'═' * 60}\n")
    print("Note: Data is real-time from mempool.space. Always cross-check with multiple explorers.")

# ────────────────────────────────────────────────
# Run the demo
# ────────────────────────────────────────────────
if __name__ == "__main__":
    test_address = "bc1qar0srrr7xfkvy5l643lydnw9re59gtzzwf5mdq"
    
    print("Validating address format...", end=" ")
    if is_valid_bitcoin_address(test_address):
        print("VALID ✓")
    else:
        print("INVALID ✗")
        exit(1)
    
    print("\nFetching data from mempool.space API...")
    stats = get_address_stats(test_address)
    print_address_summary(test_address, stats)

Validating address format... VALID ✓

Fetching data from mempool.space API...

════════════════════════════════════════════════════════════
Bitcoin Address Report (mempool.space API) - 2026-02-08 08:55 UTC
Address: bc1qar0srrr7xfkvy5l643lydnw9re59gtzzwf5mdq
════════════════════════════════════════════════════════════
Confirmed Balance : 0.16504509 BTC  (16,504,509 sat)
Total Received    : 0.16518802 BTC
Total Spent       : 0.00014293 BTC
Confirmed Tx Count: 89
Mempool Activity  : 0 pending txs

Recent Confirmed Transactions (first few):
 • 0256ca65c9d3...  confirmed=True  height=935434
 • c48cd16fade2...  confirmed=True  height=934333
 • 744a41915e63...  confirmed=True  height=933995
════════════════════════════════════════════════════════════

Note: Data is real-time from mempool.space. Always cross-check with multiple explorers.
